# Quasi-Newton Methods for Unconstrained Optimization

**Reference**: Sections 3.1 - 3.3 in Biegler (2010)

```{warning}
**AI-drafted prose, not yet reviewed by Prof. Dowling.**

Some of the writing on this page was drafted or edited by an AI assistant and has not yet been reviewed: 9 rewritten and 5 added markdown cells, measured against the last version of this notebook predating AI editing (2026-08-17).

This notice is about the *prose only*. It says nothing either way about the code, the numbers or the figures, which are checked separately. It is removed once the page has been reviewed.
```

In [1]:
# Load required Python libraries.
import numpy as np
from scipy import linalg

# Seed the random number generator so this notebook is reproducible
# (Pyomo style guide, section 8).
np.random.seed(0)

## Unconstrained Optimization with Approximate Hessian

(qn-ref-model)=
A Newton-type method uses a symmetric curvature model $\mathbf B^k$ and solves $\mathbf B^k\mathbf p^k=-\nabla f(\mathbf x^k)$. A positive-definite model gives a descent direction when the gradient is nonzero; an accepted update is $\mathbf x^{k+1}=\mathbf x^k+\alpha_k\mathbf p^k$. Unit-step modified Newton takes $\alpha_k=1$, but positive definiteness alone does not guarantee that this full step decreases the actual objective.

The examples below retain deliberate SR1 breakdowns. The detailed reference sections at the end explain the secant, signed update, curvature safeguards, and conditional rates. Biegler (2010), Algorithm 3.1 and §3.2, pp. 40–42.

For a differentiable objective and a nonzero gradient, a symmetric positive-definite model gives a descent direction. Suppose $0<m\le\lambda_j(\mathbf B)\le M$ for all $j$. Orthogonal diagonalization shows that $\mathbf B^{-1}$ has eigenvalues $1/\lambda_j(\mathbf B)$, so

$$\mathbf g^T\mathbf B^{-1}\mathbf g\ge\frac{\|\mathbf g\|^2}{M},\qquad
\|\mathbf B^{-1}\|\le\frac1m.$$

Multiplication of the first inequality by $-1$ reverses its order. The second gives the step bound by the induced matrix norm:

$$\mathbf g^T\mathbf p=-\mathbf g^T\mathbf B^{-1}\mathbf g
\le-\frac{\|\mathbf g\|^2}{M}<0,\qquad
\|\mathbf p\|=\|\mathbf B^{-1}\mathbf g\|
\le\|\mathbf B^{-1}\|\|\mathbf g\|\le\frac{\|\mathbf g\|}{m}.$$

Strict negativity uses $\mathbf g\ne\mathbf0$ and $M>0$. For $\mathbf B=\operatorname{diag}(1,4)$ and $\mathbf g=(2,1)^T$, the solve gives $\mathbf p=(-2,-1/4)^T$. Thus $\mathbf g^T\mathbf p=-4.25\le-5/4$, as the spectral bound predicts.

This proves a negative initial slope. Differentiability gives $f(\mathbf x+\alpha\mathbf p)-f(\mathbf x)=\alpha\mathbf g^T\mathbf p+o(\alpha)$, so sufficiently small positive steps decrease the objective. A full step can still overshoot. A scalar eigenvalue shift changes each eigenvalue by the same amount while preserving the eigenvectors. To enforce a floor $\epsilon>0$, the smallest nonnegative shift is $\delta=\max(0,\epsilon-\lambda_{\min})$. The diagram shows the shift geometry; the classroom example computes the shift from the original spectrum and the floor.

![A symmetric indefinite model points uphill while a shifted model gives a negative initial slope](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-direction-correction.png)

![A common shift moves the spectrum to a specified positive floor](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-eigen-shift.png)


### Modified Hessian: shift the eigenvalues

The Levenberg-Marquardt correction $\mathbf B^k=\nabla^2 f(\mathbf x^k)+\delta\mathbf I$ keeps the eigenvectors and adds $\delta$ to every eigenvalue. The smallest shift that makes every eigenvalue at least $\epsilon$ is $\delta=\max\{0,\epsilon-\lambda_{\min}\}$. The cell applies it to the lecture's eigenvalues $-2$, $1/2$ and $3$ with $\epsilon=1$.

In [2]:
import numpy as np
from scipy import linalg

# Exact Hessian with one negative eigenvalue (lecture example).
H = np.diag([-2.0, 0.5, 3.0])
eps = 1.0

# Smallest shift that makes every eigenvalue at least eps.
delta = max(0.0, eps - linalg.eigvalsh(H).min())
B = H + delta * np.eye(3)
print("delta =", delta)
print("eigenvalues of B =", linalg.eigvalsh(B))

delta = 3.0
eigenvalues of B = [1.  3.5 6. ]


SR1 update is one way to approximate $\mathbf{B}^k$.

### Library of helper functions

In [3]:
## Check if any element of array is NaN
def check_nan(A):
    return np.sum(np.isnan(A))


## Calculate gradient with central finite difference
def my_grad_approx(x, f, eps1, verbose=False):
    """
    Calculate gradient of function my_f using central difference formula

    Inputs:
        x - point for which to evaluate gradient
        f - function to consider
        eps1 - perturbation size

    Outputs:
        grad - gradient (vector)
    """

    n = len(x)
    grad = np.zeros(n)

    if verbose:
        print("***** my_grad_approx at x = ", x, "*****")

    for i in range(0, n):

        # Create vector of zeros except eps in position i
        e = np.zeros(n)
        e[i] = eps1

        # Finite difference formula
        my_f_plus = f(x + e)
        my_f_minus = f(x - e)

        # Diagnostics
        if verbose:
            print("e[", i, "] = ", e)
            print("f(x + e[", i, "]) = ", my_f_plus)
            print("f(x - e[", i, "]) = ", my_f_minus)

        grad[i] = (my_f_plus - my_f_minus) / (2 * eps1)

    if verbose:
        print("***** Done. ***** \n")

    return grad


## Calculate Hessian using central finite difference
def my_hes_approx(x, grad, eps2):
    """
    Calculate Hessian of function my_f using central difference formula and my_grad

    Inputs:
        x - point for which to evaluate gradient
        grad - function to calculate the gradient
        eps2 - perturbation size (for Hessian NOT gradient approximation)

    Outputs:
        H - Hessian (matrix)
    """

    n = len(x)
    H = np.zeros([n, n])

    for i in range(0, n):
        # Create vector of zeros except eps in position i
        e = np.zeros(n)
        e[i] = eps2

        # Evaluate gradient twice
        grad_plus = grad(x + e)
        grad_minus = grad(x - e)

        # Notice we are building the Hessian by column (or row)
        H[:, i] = (grad_plus - grad_minus) / (2 * eps2)

    return H


## Linear algebra calculation
def xxT(u):
    """
    Calculates u*u.T to circumvent limitation with SciPy

    Arguments:
    u - numpy 1D array

    Returns:
    u*u.T

    Assume u is a nx1 vector.
    Recall: NumPy does not distinguish between row or column vectors

    u.dot(u) returns a scalar. This function returns an nxn matrix.
    """

    n = len(u)
    A = np.zeros([n, n])
    for i in range(0, n):
        for j in range(0, n):
            A[i, j] = u[i] * u[j]

    return A


## Analyze Hessian
def analyze_hes(B):
    print(B, "\n")

    l = linalg.eigvals(B)
    print("Eigenvalues: ", l, "\n")

### Symmetric Rank 1 (SR1) Update

In [4]:
def alg1_sr1(x0, calc_f, calc_grad, eps1, eps2, verbose=False, max_iter=1000):
    """
    Arguments:
        x0 - starting point
        calc_f - function that calculates f(x)
        calc_grad - function that calculates gradient(x)

    Outputs:
        x - iteration history of x (decision variables)
        f - iteration history of f(x) (objective value)
        p - iteration history of p (steps)
        B - Hessian approximation
    """

    # Allocate outputs as empty lists
    x = []
    f = []
    p = []
    grad = []
    B = []

    # Store starting point
    x.append(x0)
    k = 0

    flag = True

    print("Iter. \tf(x) \t\t||grad(x)|| \t||p|| \t\tmin(lambda)")

    while flag and k < max_iter:
        # Evaluate f(x) at current iteration
        f.append(calc_f(x[k]))

        # Evaluate gradient
        grad.append(calc_grad(x[k]))

        if check_nan(grad[k]):
            print("WARNING: gradient calculation returned NaN")
            break

        if verbose:
            print("\n")
            print("k = ", k)
            print("x = ", x[k])
            print("grad = ", grad[k])

        # Update Hessian approximation
        if k == 0:
            # Initialize with identity
            B.append(np.eye(len(x0)))

        else:
            # Change in x
            s = x[k] - x[k - 1]

            # Change in gradient
            y = grad[k] - grad[k - 1]

            # SR1 formulation
            u = y - B[k - 1].dot(s)
            denom = (u).dot(s)

            # Safeguard: skip test for the SR1 update.
            # Only apply the update when
            #     | (y - B^k s)^T s |  >=  C1 * ||s||^2,   C1 > 0
            # and otherwise leave B unchanged. See the "Quasi-Newton Methods"
            # lecture notes, Strategy 1 (SR1), and Biegler (2010), p. 43.
            # If the denominator is (near) zero the rank-one correction is an
            # arbitrarily large, arbitrarily ill-conditioned change built from
            # almost no new information. Note the test is two-sided: it looks at
            # the magnitude only, because -- unlike BFGS -- SR1 is allowed to
            # represent negative curvature.
            C1 = 1e-8

            if abs(denom) >= C1 * s.dot(s):
                # Formula: dB = u * u.T / (u.T * s) if u is a column vector.
                dB = xxT(u) / denom
                skipped = False
            else:
                # Skip the update: B[k] = B[k-1]
                dB = np.zeros((len(x0), len(x0)))
                skipped = True

            if verbose:
                print("s = ", s)
                print("y = ", y)
                print("SR1 update denominator, (y-B[k-1]*s).T*s = ", denom)
                print("SR1 update u = ", u)
                if skipped:
                    print("Skip test |denom| >= C1*||s||^2 failed: update skipped.")
                print("SR1 update u.T*u/(u.T*s) = \n", dB)

            B.append(B[k - 1] + dB)

        if verbose:
            print("B = \n", B[k])

        if check_nan(B[k]):
            print("WARNING: Hessian update returned NaN")
            break

        c = np.linalg.cond(B[k])
        if c > 1e12:
            flag = False
            print("Warning: Hessian approximation is near singular.")
            print("B[k] = \n", B[k])

        else:
            # Calculate step
            p.append(linalg.solve(B[k], -grad[k]))

            if verbose:
                print("p = ", p[k])

            # Take step
            x.append(x[k] + p[k])

            # Calculate norms
            norm_p = linalg.norm(p[k])
            norm_g = linalg.norm(grad[k])

            # Calculate eigenvalues of Hessian (for display only)
            ev = np.real(linalg.eigvals(B[k]))

            # print("k = ",k,"\t"f[k],"\t",norm_g,"\t",norm_p)
            print(
                k,
                "  \t{0: 1.4e} \t{1:1.4e} \t{2:1.4e} \t{3: 1.4e}".format(
                    f[k], norm_g, norm_p, np.min(ev)
                ),
            )

            # Check convergence criteria
            flag = (norm_p > eps1) and (norm_g > eps2)

            # Update iteration counter
            k = k + 1

    print("Done.")
    print("x* = ", x[-1])

    return x, f, p, B


print("Done.")

Done.


### SR1 update as a short function

`alg1_sr1` above builds the SR1 update inline. The function below isolates the formula and a scale-aware skip test, $|\mathbf u^T\mathbf s|\ge r\|\mathbf u\|\|\mathbf s\|$ with $\mathbf u=\mathbf y-\mathbf B\mathbf s$ (Nocedal & Wright, Eq. (6.26)), then checks the secant equation $\mathbf B^{k+1}\mathbf s=\mathbf y$ on the lecture's data.

In [5]:
import numpy as np
from scipy import linalg


def sr1_update(B, s, y, r=1e-8):
    """Symmetric rank-one (SR1) update of the Hessian approximation B."""
    u = y - B @ s  # secant residual
    # Skip test: keep B when |u^T s| is small relative to ||u|| ||s||.
    # This also covers u = 0, when B already satisfies the secant equation.
    if abs(u @ s) <= r * linalg.norm(u) * linalg.norm(s):
        return B
    return B + np.outer(u, u) / (u @ s)


# Lecture data: one secant pair (s, y), starting from B = I.
B = np.eye(2)
s = np.array([1.0, 0.0])
y = np.array([2.0, 1.0])
B_sr1 = sr1_update(B, s, y)
print("SR1 update:\n", B_sr1)
print("B @ s =", B_sr1 @ s)

SR1 update:
 [[2. 1.]
 [1. 2.]]
B @ s = [2. 1.]


## Test Case: Simple quadratic program

$$\min_{\mathbf{x}} ~~ x_1^2 + (x_2 -1)^2$$

In [6]:
def my_f_simple(x):
    return x[0] ** 2 + (x[1] - 1) ** 2


def my_grad_exact(x):
    return np.array([2 * x[0], 2 * (x[1] - 1)])

### Near solution
Consider $\mathbf{x}_0 = [-0.1, 0.2]^T$

In [7]:
# Specify convergence criteria
eps1 = 1e-8
eps2 = 1e-4

# Create a Lambda (anonymous) function for gradient calculation
# calc_grad = lambda x : my_grad_approx(x,my_f_simple,1E-6,verbose=False)
calc_grad = lambda x: my_grad_exact(x)

# Specify starting point
x0 = np.array([-0.1, 0.2])

# Call optimization routine
x, f, p, B = alg1_sr1(
    x0, my_f_simple, calc_grad, eps1, eps2, verbose=False, max_iter=50
)

# SR1 Hessian approximation
print("\nSR1 Hessian approximation. B[k] =")
analyze_hes(B[-1])

# Actual Hessian
print("True Hessian at x*. B =")
analyze_hes(my_hes_approx(x[-1], calc_grad, 1e-6))

Iter. 	f(x) 		||grad(x)|| 	||p|| 		min(lambda)
0   	 6.5000e-01 	1.6125e+00 	1.6125e+00 	 1.0000e+00
1   	 6.5000e-01 	1.6125e+00 	8.0623e-01 	 1.0000e+00
2   	 0.0000e+00 	0.0000e+00 	0.0000e+00 	 1.0000e+00
Done.
x* =  [0. 1.]

SR1 Hessian approximation. B[k] =
[[1.01538462 0.12307692]
 [0.12307692 1.98461538]] 

Eigenvalues:  [1.+0.j 2.+0.j] 

True Hessian at x*. B =
[[2. 0.]
 [0. 2.]] 

Eigenvalues:  [2.+0.j 2.+0.j] 



### Far from solution
Consider $\mathbf{x}_0 = [-100, 500]^T$

In [8]:
# Specify starting point
x0 = np.array([-100, 500])

# Call optimization routine
x, f, p, B = alg1_sr1(
    x0, my_f_simple, calc_grad, eps1, eps2, verbose=False, max_iter=50
)

# SR1 Hessian approximation
print("\nSR1 Hessian approximation. B[k] =")
analyze_hes(B[-1])

# Actual Hessian
print("True Hessian at x*. B =")
analyze_hes(my_hes_approx(x[-1], calc_grad, 1e-6))

Iter. 	f(x) 		||grad(x)|| 	||p|| 		min(lambda)
0   	 2.5900e+05 	1.0178e+03 	1.0178e+03 	 1.0000e+00
1   	 2.5900e+05 	1.0178e+03 	5.0892e+02 	 1.0000e+00
2   	 5.0487e-27 	1.4211e-13 	1.1604e-13 	 1.0000e+00
Done.
x* =  [-5.1938169e-14  1.0000000e+00]

SR1 Hessian approximation. B[k] =
[[ 1.03860989 -0.19266335]
 [-0.19266335  1.96139011]] 

Eigenvalues:  [1.+0.j 2.+0.j] 

True Hessian at x*. B =
[[2. 0.]
 [0. 2.]] 

Eigenvalues:  [2.+0.j 2.+0.j] 



### Activity/Discussion
* Does the number of iterations depend on the starting point for this problem?
* How many iterations are needed for Newton's method to converge for a positive definite quadratic program using exact second derivative information?
* Why does the SR1 update not converge to the true Hessian?

## Test Case: Example 2.19

Biegler's two-variable test problem is

$$\min_{\mathbf{x}} f(\mathbf{x})=\alpha e^{-\beta},$$

with $u=x_1-0.8$, $v=x_2-(a_1+a_2u^2(1-u)^{1/2}-a_3u)$,
$\alpha=-b_1+b_2u^2(1+u)^{1/2}+b_3u$, and
$\beta=c_1v^2(1-c_2v)/(1+c_3u^2)$. We use
$\mathbf{a}=(0.3,0.6,0.2)$, $\mathbf{b}=(5,26,3)$, and $\mathbf{c}=(40,1,10)$.

The reported solution is $\mathbf{x}^*=(0.7395,0.3144)$ with $f(\mathbf{x}^*)=-5.0893$.
The cells below recompute it rather than relying on the printed result.
(Biegler, Example 2.19, p. 30.)

In [9]:
## Define Python function to calculate objective
def my_f_ex_2_19(x, verbose=False):
    """Evaluate function given above at point x

    Inputs:
        x - vector with 2 elements

    Outputs:
        f - function value (scalar)
    """
    # Constants
    a = np.array([0.3, 0.6, 0.2])
    b = np.array([5, 26, 3])
    c = np.array([40, 1, 10])

    # Intermediates. Recall Python indicies start at 0
    u = x[0] - 0.8
    s = np.sqrt(1 - u)
    s2 = np.sqrt(1 + u)
    v = x[1] - (a[0] + a[1] * u**2 * s - a[2] * u)
    alpha = (
        -b[0] + b[1] * u**2 * s2 + b[2] * u
    )  # September 5, 2018: changed 's' to 's2'
    beta = c[0] * v**2 * (1 - c[1] * v) / (1 + c[2] * u**2)
    f = alpha * np.exp(-beta)

    if verbose:
        print("##### my_f at x = ", x, "#####")
        print("u = ", u)
        print("sqrt(1-u) = ", s)
        print("sqrt(1+u) = ", s2)
        print("v = ", v)
        print("alpha = ", alpha)
        print("beta = ", beta)
        print("f(x) = ", f)
        print("##### Done. #####\n")

    return f

### $\mathbf{x}_0$ somewhat near solution

Consider:
$$\mathbf{x}_0 = [0.3, 0.1]^T$$

In [10]:
# Create a Lambda (anonymous) function for gradient calculation
calc_grad = lambda x: my_grad_approx(x, my_f_ex_2_19, 1e-6, verbose=False)

# Specify starting point
x0 = np.array([0.3, 0.1])

# Call optimization routine
x, f, p, B = alg1_sr1(
    x0, my_f_ex_2_19, calc_grad, eps1, eps2, verbose=False, max_iter=250
)

# SR1 Hessian approximation
print("\nSR1 Hessian approximation. B[k] =")
analyze_hes(B[-1])

# Actual Hessian
print("True Hessian at x*. B =")
analyze_hes(my_hes_approx(x[-1], calc_grad, 1e-6))

Iter. 	f(x) 		||grad(x)|| 	||p|| 		min(lambda)
0   	-3.6022e-02 	8.3847e-01 	8.3847e-01 	 1.0000e+00
1   	-4.1276e-02 	4.7785e-01 	3.0223e-01 	 1.0000e+00
2   	-2.2301e+00 	2.1236e+01 	3.0865e-01 	-6.8856e+01
3   	-4.1260e-02 	4.4645e-01 	1.3053e-01 	-6.7478e+01
4   	-1.0354e-01 	1.1855e+00 	1.5062e-01 	-7.1805e+01
5   	-3.7115e-02 	3.9982e-01 	1.3096e-02 	-5.8783e+01
6   	-3.4039e-02 	3.6164e-01 	5.9865e-02 	-1.4365e+01
7   	-2.2398e-02 	2.0377e-01 	4.2988e-02 	-9.5209e+00
8   	-1.7133e-02 	1.1914e-01 	4.2522e-02 	-6.4394e+00
9   	-1.4292e-02 	5.7282e-02 	2.4810e-02 	-4.8015e+00
10   	-1.3542e-02 	2.5813e-02 	1.0353e-02 	-2.8612e+00
11   	-1.3373e-02 	1.0933e-02 	8.4091e-03 	-1.6117e+00
12   	-1.3325e-02 	7.7765e-04 	7.5906e-04 	-1.6030e+00
13   	-1.3324e-02 	1.7106e-05 	1.4129e-05 	-1.5761e+00
Done.
x* =  [0.80557705 0.96556999]

SR1 Hessian approximation. B[k] =
[[-1.48333926 -0.21589308]
 [-0.21589308 -1.07333657]] 

Eigenvalues:  [-1.57605485+0.j -0.98062098+0.j] 

True Hessian at

### $\mathbf{x}_0$ far from solution

Consider:
$$\mathbf{x}_0 = [-0.1, 0.2]^T$$

In [11]:
# Specify starting point
x0 = np.array([-0.1, 0.2])

# Call optimization routine
x, f, p, B = alg1_sr1(
    x0, my_f_ex_2_19, calc_grad, eps1, eps2, verbose=False, max_iter=250
)

# SR1 Hessian approximation
print("\nSR1 Hessian approximation. B[k] =")
analyze_hes(B[-1])

# Actual Hessian
print("True Hessian at x*. B =")
analyze_hes(my_hes_approx(x[-1], calc_grad, 1e-6))

Iter. 	f(x) 		||grad(x)|| 	||p|| 		min(lambda)
0   	-4.5540e-04 	9.2580e-03 	9.2580e-03 	 1.0000e+00
1   	-5.4879e-04 	1.0959e-02 	5.9578e-02 	-1.8395e-01
2   	-1.5604e-04 	3.4877e-03 	2.7627e-02 	-1.2568e-01
3   	-8.3235e-05 	1.9563e-03 	3.5000e-02 	-5.5390e-02
4   	-3.6006e-05 	9.0023e-04 	2.9559e-02 	-3.0011e-02
5   	-1.7097e-05 	4.4964e-04 	2.9246e-02 	-1.5078e-02
6   	-7.9092e-06 	2.1835e-04 	2.7389e-02 	-7.7836e-03
7   	-3.7242e-06 	1.0745e-04 	2.6349e-02 	-3.9664e-03
8   	-1.7531e-06 	5.2702e-05 	2.5208e-02 	-2.0268e-03
Done.
x* =  [-0.11328951 -0.05033867]

SR1 Hessian approximation. B[k] =
[[ 1.00141633e+00 -4.82365648e-02]
 [-4.82365648e-02  2.91945446e-04]] 

Eigenvalues:  [ 1.00373512+0.j -0.00202684+0.j] 

True Hessian at x*. B =
[[ 6.99095467e-05 -2.18304769e-04]
 [-2.18304769e-04 -7.02715474e-04]] 

Eigenvalues:  [ 0.00012732+0.j -0.00076013+0.j] 



### Activity/Discussion
* Classify each candidate solution.
* Is the SR1 approximation always positive definite?

## Broyden update with Cholesky factorization
As part of Algorithms Homework 3, you will adapt this example and implement the BFGS (Broyden-Fletcher-Goldfarb-Shanno) update formula. 


You may decide to use the Cholesky factorization of $\mathbf{B}^{k}$,

$$\mathbf{B}^{k} = \mathbf{L}^{k} (\mathbf{L}^{k})^T,$$

to make your BFGS update more efficient. (This is not required). Let's see how to do Cholesky factorization with SciPy.

In [12]:
## Create random P.D. and symmetric matrix
B_ = np.eye(3) + xxT(np.random.normal(0, 1, 3))
print("B = \n", B_, "\n")

## Perform Cholesky factorization.
# By default, lower=False and L is upper triangular. Either works here,
# but we prefer L to be lower triangular for convention.
L = linalg.cholesky(B_, lower=True)
print("L = \n", L, "\n")

## Reconstruct B
print("L*L.T = \n", L.dot(L.T), "\n")

B = 
 [[4.11188068 0.70589826 1.72654504]
 [0.70589826 1.16012579 0.39164906]
 [1.72654504 0.39164906 1.95792804]] 

L = 
 [[2.02777728 0.         0.        ]
 [0.3481143  1.01928515 0.        ]
 [0.85144708 0.09344604 1.10645097]] 

L*L.T = 
 [[4.11188068 0.70589826 1.72654504]
 [0.70589826 1.16012579 0.39164906]
 [1.72654504 0.39164906 1.95792804]] 



### BFGS update as a short function

The function below implements the BFGS update with the skip test $\mathbf s^T\mathbf y\ge C_2\|\mathbf s\|^2$ and applies it to the same secant pair as the SR1 cell. Both updates satisfy $\mathbf B^{k+1}\mathbf s=\mathbf y$; they differ in the curvature assigned to the direction $\mathbf e_2$, which this step did not test.

In [13]:
import numpy as np


def bfgs_update(B, s, y, C2=1e-8):
    """BFGS update of the Hessian approximation B."""
    # Skip test: keep B unless the curvature s^T y >= C2 ||s||^2.
    if s @ y < C2 * (s @ s):
        return B
    Bs = B @ s
    return B + np.outer(y, y) / (s @ y) - np.outer(Bs, Bs) / (s @ Bs)


# Same secant pair as the SR1 cell, starting from B = I.
B = np.eye(2)
s = np.array([1.0, 0.0])
y = np.array([2.0, 1.0])
B_bfgs = bfgs_update(B, s, y)
print("BFGS update:\n", B_bfgs)
print("B @ s =", B_bfgs @ s)

BFGS update:
 [[2.  1. ]
 [1.  1.5]]
B @ s = [2. 1.]


(qn-ref-unit-steps)=
## Example: four Newton-type methods with unit steps

The cells below apply Algorithm 16.1 with unit steps, $\alpha_k=1$, and no line search, as in `alg1_sr1` above. The test problem is a gentle version of the Rosenbrock function,

$$f(\mathbf x)=(x_2-x_1^2)^2+(1-x_1)^2,\qquad \mathbf x^0=(-1.2,1)^T,\qquad \mathbf x^*=(1,1)^T,$$

with $\nabla^2 f(\mathbf x^*)\succ0$. The standard Rosenbrock function multiplies the first term by 100; the next lecture uses it to show why unit steps need a line search.

`newton_type` either uses the exact Hessian, shifted so every eigenvalue is at least $\epsilon$ (modified Newton), or applies a quasi-Newton update starting from $\mathbf B^0$. The columns are the iteration, $f(\mathbf x^k)$, $\|\nabla f(\mathbf x^k)\|$, the slope $\nabla f(\mathbf x^k)^T\mathbf p^k$, the smallest eigenvalue of $\mathbf B^k$, the error $\|\mathbf x^k-\mathbf x^*\|$, and the directional error $\|(\mathbf B^k-\nabla^2 f(\mathbf x^k))\mathbf p^k\|/\|\mathbf p^k\|$.

In [14]:
import numpy as np
from scipy import linalg


def gentle_rosenbrock(x):
    return (x[1] - x[0] ** 2) ** 2 + (1.0 - x[0]) ** 2


def gentle_rosenbrock_grad(x):
    return np.array(
        [-4.0 * x[0] * (x[1] - x[0] ** 2) - 2.0 * (1.0 - x[0]),
         2.0 * (x[1] - x[0] ** 2)]
    )


def gentle_rosenbrock_hess(x):
    return np.array(
        [[-4.0 * (x[1] - 3.0 * x[0] ** 2) + 2.0, -4.0 * x[0]],
         [-4.0 * x[0], 2.0]]
    )

In [15]:
def newton_type(x0, calc_f, calc_grad, calc_hess, x_star, update=None,
                B0=None, eps=1e-3, max_iter=100, gtol=1e-8):
    """Algorithm 16.1 with unit steps (alpha = 1) and console output."""
    x = np.array(x0, dtype=float)
    B = np.eye(len(x)) if B0 is None else B0  # starting approximation
    print("Iter. f(x)        ||grad(x)|| g^T p        min eig(B)   ||x-x*||    "
          "dir. error")
    for k in range(max_iter):
        g = calc_grad(x)
        if linalg.norm(g) <= gtol:
            print(f"Converged at iteration {k}: ||grad(x)|| <= {gtol:.0e}.")
            break
        H = calc_hess(x)  # used for modified Newton and the dir. error column
        if update is None:
            # Modified Newton: shift the exact Hessian (Levenberg-Marquardt).
            B = H + max(0.0, eps - linalg.eigvalsh(H).min()) * np.eye(len(x))
        p = linalg.solve(B, -g)
        dir_err = linalg.norm((B - H) @ p) / linalg.norm(p)
        print(f"{k:<5d} {calc_f(x):.4e}  {linalg.norm(g):.4e}  {g @ p: .4e}  "
              f"{linalg.eigvalsh(B).min(): .4e}  {linalg.norm(x - x_star):.4e}  "
              f"{dir_err:.4e}")
        x_new = x + p  # unit step: no line search
        if update is not None:
            B = update(B, p, calc_grad(x_new) - g)  # s = p, y = gradient change
        x = x_new
    else:
        print(f"Stopped after {max_iter} iterations without convergence.")
    return x, B

### Modified Newton (quadratic convergence)

In [16]:
x0 = np.array([-1.2, 1.0])
x_star = np.array([1.0, 1.0])
x, B = newton_type(x0, gentle_rosenbrock, gentle_rosenbrock_grad,
                   gentle_rosenbrock_hess, x_star)

Iter. f(x)        ||grad(x)|| g^T p        min eig(B)   ||x-x*||    dir. error
0     5.0336e+00  6.5712e+00  -5.5361e+00   4.4673e-01  2.2000e+00  0.0000e+00
1     2.9357e+00  3.5273e+00  -4.3178e+00   1.9974e+00  2.5827e+00  0.0000e+00
2     5.7481e-01  1.4422e+00  -9.9968e-01   1.3348e+00  1.2650e+00  0.0000e+00
3     1.9391e-01  1.5968e+00  -3.7870e-01   6.4278e-01  6.2581e-01  0.0000e+00
4     2.1142e-03  8.1066e-02  -4.2044e-03   3.6820e-01  1.0329e-01  0.0000e+00
5     4.4032e-06  8.8455e-03  -8.8059e-06   3.4449e-01  2.6172e-03  0.0000e+00
6     1.1767e-12  1.8997e-06  -2.3534e-12   3.4315e-01  2.4815e-06  0.0000e+00
Converged at iteration 7: ||grad(x)|| <= 1e-08.


### BFGS (superlinear convergence)

In [17]:
x, B = newton_type(x0, gentle_rosenbrock, gentle_rosenbrock_grad,
                   gentle_rosenbrock_hess, x_star, update=bfgs_update)

Iter. f(x)        ||grad(x)|| g^T p        min eig(B)   ||x-x*||    dir. error
0     5.0336e+00  6.5712e+00  -4.3181e+01   1.0000e+00  2.2000e+00  1.5582e+01
1     7.1225e+02  5.7068e+02  -3.6136e+03   9.5005e-01  4.4009e+00  2.4410e+02
2     6.4052e+00  3.8407e+00  -1.1778e+01   9.5015e-01  2.5137e+00  3.9551e+00
3     1.3269e+01  2.0513e+01  -1.8730e+01   2.7507e+00  3.0581e+00  5.0476e-01
4     4.2965e+00  3.1000e+00  -1.3825e-01   2.9673e+00  2.0904e+00  2.7935e+01
5     4.1622e+00  3.3040e+00  -3.8355e+00   4.5628e-01  2.0470e+00  2.1581e+00
6     3.4493e+00  4.3995e+00  -3.0481e+00   1.0996e+00  2.6820e+00  3.4545e+00
7     2.4263e+00  3.5923e+00  -5.7943e-01   1.5833e+00  1.7998e+00  9.8470e+00
8     1.9078e+00  3.1745e+00  -2.0617e+00   1.1670e+00  1.7925e+00  1.0441e+00
9     1.3078e+00  2.0772e+00  -1.3975e+00   3.0744e+00  1.8914e+00  3.2718e+00
10    6.6750e-01  1.4596e+00  -3.4077e-01   2.3310e+00  1.4105e+00  3.6549e+00
11    3.8287e-01  1.0571e+00  -5.4164e-01   1.3146e+

### SR1

SR1 can produce an indefinite $\mathbf B^k$. At such an iterate the computed step can be an ascent direction, $\nabla f^T\mathbf p>0$, and the unit step is taken anyway.

In [18]:
x, B = newton_type(x0, gentle_rosenbrock, gentle_rosenbrock_grad,
                   gentle_rosenbrock_hess, x_star, update=sr1_update)

Iter. f(x)        ||grad(x)|| g^T p        min eig(B)   ||x-x*||    dir. error
0     5.0336e+00  6.5712e+00  -4.3181e+01   1.0000e+00  2.2000e+00  1.5582e+01
1     7.1225e+02  5.7068e+02  -3.6135e+03   1.0000e+00  4.4009e+00  2.4423e+02
2     6.1579e+00  3.5078e+00  -2.4097e+02   4.0595e-02  2.4731e+00  4.9112e+00
3     2.1751e+04  5.0563e+03  -6.0438e+04   1.8381e+00  7.5809e+01  8.5297e+01
4     3.7942e+00  3.9320e+00  -3.6134e-02   1.8378e+00  1.9510e+00  7.1682e+01
5     3.7596e+00  4.0562e+00  -5.0521e+00   4.7947e-01  1.9456e+00  5.6264e-01
6     3.8458e+00  3.9898e+00  -8.4814e+00   1.8768e+00  2.7939e+00  4.5060e+00
7     1.9622e+00  2.6355e+00  -7.7284e-01   2.3123e+00  1.5796e+00  5.5744e+00
8     1.2852e+00  2.3160e+00  -1.9499e+00   1.1800e+00  1.5610e+00  1.0091e+00
9     1.8428e+00  4.5303e+00  -3.8732e+00   4.4580e+00  1.7609e+00  6.0991e+00
10    6.6382e-01  1.5361e+00  -2.2287e-01   4.4699e+00  1.3504e+00  8.0617e+00
11    4.7078e-01  1.1343e+00  -5.4501e-01   2.2307e+

### Steepest descent (linear convergence)

Keeping $\mathbf B^k=10\mathbf I$ fixed gives the steepest-descent step $-\nabla f/10$.

In [19]:
def no_update(B, s, y):
    return B  # keep B = 10 I: steepest descent with a fixed step


x, B = newton_type(x0, gentle_rosenbrock, gentle_rosenbrock_grad,
                   gentle_rosenbrock_hess, x_star, update=no_update,
                   B0=10.0 * np.eye(2), max_iter=1000)

Iter. f(x)        ||grad(x)|| g^T p        min eig(B)   ||x-x*||    dir. error
0     5.0336e+00  6.5712e+00  -4.3181e+00   1.0000e+01  2.2000e+00  6.9355e+00
1     3.0179e+00  2.0867e+00  -4.3543e-01   1.0000e+01  1.5513e+00  1.0514e+01
2     2.5723e+00  2.1867e+00  -4.7818e-01   1.0000e+01  1.4135e+00  1.0546e+01
3     2.0835e+00  2.2833e+00  -5.2134e-01   1.0000e+01  1.2742e+00  1.0457e+01
4     1.5556e+00  2.3301e+00  -5.4295e-01   1.0000e+01  1.1365e+00  1.0078e+01
5     1.0208e+00  2.2301e+00  -4.9733e-01   1.0000e+01  1.0078e+00  9.1766e+00
6     5.5867e-01  1.8611e+00  -3.4637e-01   1.0000e+01  9.0039e-01  7.6572e+00
7     2.6599e-01  1.2339e+00  -1.5224e-01   1.0000e+01  8.2407e-01  6.0608e+00
8     1.4979e-01  6.4736e-01  -4.1908e-02   1.0000e+01  7.7365e-01  6.2999e+00
9     1.1764e-01  3.9027e-01  -1.5231e-02   1.0000e+01  7.3468e-01  8.5752e+00
10    1.0383e-01  3.3377e-01  -1.1140e-02   1.0000e+01  6.9938e-01  9.2959e+00
11    9.3096e-02  3.1052e-01  -9.6422e-03   1.0000e+

(qn-ref-secant)=
## Reference: secant measurements and signed SR1

Use Euclidean vector norms and their induced matrix norm unless a weighted norm is specified.

For a nonzero accepted displacement, $\mathbf s=\mathbf x^{k+1}-\mathbf x^k$ and $\mathbf y=\nabla f(\mathbf x^{k+1})-\nabla f(\mathbf x^k)$. Assume $f\in C^2(D)$ on an open $D$ containing the full segment $\{\mathbf x^k+t\mathbf s:0\le t\le1\}$. Define $\mathbf h(t)=\nabla f(\mathbf x^k+t\mathbf s)$. The chain rule gives

$$\mathbf h'(t)=\nabla^2f(\mathbf x^k+t\mathbf s)\mathbf s.$$

Apply the fundamental theorem of calculus to each component. Since $\mathbf s$ is constant in $t$, it can be factored outside the integral:

$$\mathbf y=\mathbf h(1)-\mathbf h(0)
=\int_0^1\nabla^2f(\mathbf x^k+t\mathbf s)\mathbf s\,dt
=\left[\int_0^1\nabla^2f(\mathbf x^k+t\mathbf s)\,dt\right]\mathbf s.$$

This exact average-Hessian identity motivates $\mathbf B^{k+1}\mathbf s=\mathbf y$. It fixes the action on one direction, not every matrix entry. A symmetric $n\times n$ matrix has $n(n+1)/2$ free entries and a nonzero secant gives $n$ independent scalar equations. For example, $\mathbf s=\mathbf e_1$, $\mathbf y=(2,1)^T$ gives $\mathbf B^+=\begin{bmatrix}2&1\\1&c\end{bmatrix}$ with free $c$; positive definiteness adds $c>1/2$.

For $\mathbf u=\mathbf y-\mathbf B\mathbf s$, signed SR1 is

$$\mathbf B^+=\mathbf B+\frac{\mathbf u\mathbf u^T}{\mathbf u^T\mathbf s},\qquad \mathbf u^T\mathbf s\ne0.$$

To derive the sign and scale, start with $\mathbf B^+=\mathbf B+\sigma\mathbf w\mathbf w^T$. The secant equation requires $\sigma\mathbf w(\mathbf w^T\mathbf s)=\mathbf u$. Since $\mathbf u\ne0$, the correction direction is parallel to $\mathbf u$; write $\mathbf w=a\mathbf u$. Substitution gives

$$\sigma a^2(\mathbf u^T\mathbf s)\mathbf u=\mathbf u,
\qquad \sigma a^2(\mathbf u^T\mathbf s)=1.$$

Choose $\sigma=\operatorname{sign}(\mathbf u^T\mathbf s)$ and $a=1/\sqrt{|\mathbf u^T\mathbf s|}$, giving $\mathbf w=\mathbf u/\sqrt{|\mathbf u^T\mathbf s|}$. A formula using only $+\mathbf w\mathbf w^T$ cannot represent negative denominators. Multiplying by $\mathbf s$ cancels the denominator and gives $\mathbf B^+\mathbf s=\mathbf B\mathbf s+\mathbf u=\mathbf y$.

To see why the zero-denominator case is an obstruction, suppose a nonzero symmetric rank-one correction $\sigma\mathbf w\mathbf w^T$ satisfies $\sigma\mathbf w(\mathbf w^T\mathbf s)=\mathbf u\ne0$. Then $\mathbf u$ is parallel to $\mathbf w$ and $\mathbf w^T\mathbf s\ne0$, so $\mathbf u^T\mathbf s=\sigma(\mathbf w^T\mathbf s)^2\ne0$. Thus $\mathbf u\ne0$ with $\mathbf u^T\mathbf s=0$ cannot be repaired by that rank-one form.

There are three cases: if $\mathbf u=0$, keep the existing matrix; if $\mathbf u\ne0$ but $\mathbf u^T\mathbf s=0$, no symmetric rank-one secant correction exists; if the denominator is small but nonzero, a large correction can amplify errors. Nocedal and Wright's scale-aware safeguard accepts only if

$$|\mathbf u^T\mathbf s|\ge r\|\mathbf u\|\|\mathbf s\|,\quad0<r<1.$$

Biegler gives the alternative threshold $|\mathbf u^T\mathbf s|\ge C_1\|\mathbf s\|^2$. Neither test enforces a positive sign; SR1 can represent indefinite curvature. A trust-region method can use that curvature, whereas a basic line-search step requires a descent direction.

Direction coverage also matters for recovering a Hessian. The precise nonlinear SR1 result uses **uniform** linear independence and its other convergence/safeguard hypotheses, not merely a different-looking sequence of steps. See Nocedal and Wright, Theorem 6.2, p. 149. Biegler's short independence remark on p. 43 should be read with those hypotheses.

Sources: Biegler (2010), Eqs. (3.8)–(3.10), pp. 42–43; Nocedal and Wright (2006), §6.2, Eqs. (6.24) and (6.26), pp. 144–149. 

![A scalar secant measures change in gradient over accepted displacement](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-secant.png)

![The same measured direction permits different symmetric matrix actions elsewhere](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-secant-family.png)

![Positive and negative secant projections determine the sign of the rank-one correction](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-sr1-geometry.png)


(qn-ref-bfgs)=
## Reference: BFGS, curvature safeguards, and implementation alternatives

With $\mathbf B\succ0$, $\mathbf s\ne\mathbf0$ and $\mathbf s^T\mathbf y>0$,

$$\mathbf B^+=\mathbf B+\frac{\mathbf y\mathbf y^T}{\mathbf s^T\mathbf y}
-\frac{\mathbf B\mathbf s\mathbf s^T\mathbf B}{\mathbf s^T\mathbf B\mathbf s}.$$

Multiplication by $\mathbf s$ gives $\mathbf B\mathbf s+\mathbf y-\mathbf B\mathbf s=\mathbf y$. The correction has rank at most two. The positive curvature condition is necessary for any positive-definite secant matrix: $\mathbf s^T\mathbf y=\mathbf s^T\mathbf B^+\mathbf s>0$.

A stronger quantitative test $\mathbf s^T\mathbf y\ge C_2\|\mathbf s\|^2$ can reject **small positive** curvature as well as zero or negative curvature. It does not, by itself, bound the matrix norm. With $\mathbf B=\mathbf I$, $\mathbf s=\mathbf e_1$, and $\mathbf y=(1,a)^T$, the scalar test is always $\mathbf s^T\mathbf y=1$, but

$$\mathbf B^+=\begin{bmatrix}1&a\\a&1+a^2\end{bmatrix}$$

has an arbitrarily large largest eigenvalue as $|a|$ grows. Additional bounds on measurements and updates matter.

**Powell damping.** Let $b=\mathbf s^T\mathbf B\mathbf s>0$ and choose $\beta\in(0,1)$, commonly $0.2$. Define $\bar{\mathbf y}=\theta\mathbf y+(1-\theta)\mathbf B\mathbf s$. Taking

$$\theta=\begin{cases}1,&\mathbf s^T\mathbf y\ge\beta b,\\
\dfrac{(1-\beta)b}{b-\mathbf s^T\mathbf y},&\mathbf s^T\mathbf y<\beta b\end{cases}$$

makes $\mathbf s^T\bar{\mathbf y}\ge\beta b$; in the second case equality holds. The update then fits the damped measurement. For the handout's example $\mathbf s=\mathbf e_1$, $\mathbf B\mathbf s=(1,0)^T$, $\mathbf y=(-1,1)^T$, and $\beta=0.2$, take the inner product of the mixture with $\mathbf s$:

$$\mathbf s^T\bar{\mathbf y}=\theta(-1)+(1-\theta)(1)=1-2\theta.$$

The constraint $1-2\theta\ge0.2$ gives $\theta\le0.4$. The largest allowed weight, $\theta=0.4$, yields $\bar{\mathbf y}=0.4(-1,1)^T+0.6(1,0)^T=(0.2,0.4)^T$. The prepared picture below shows the interpolation and the permitted curvature half-space.

**Inverse form.** For $\mathbf H=\mathbf B^{-1}$, $\rho=1/(\mathbf s^T\mathbf y)$,

$$\mathbf H^+=(\mathbf I-\rho\mathbf s\mathbf y^T)\mathbf H
(\mathbf I-\rho\mathbf y\mathbf s^T)+\rho\mathbf s\mathbf s^T.$$

Assume $\mathbf H\succ0$ and $\rho>0$. For any nonzero $\mathbf z$, write $\mathbf w=(\mathbf I-\rho\mathbf y\mathbf s^T)\mathbf z$. The two matrix factors on either side of $\mathbf H$ are transposes, so

$$\mathbf z^T\mathbf H^+\mathbf z=\mathbf w^T\mathbf H\mathbf w+\rho(\mathbf s^T\mathbf z)^2.$$

Both terms are nonnegative. If $\mathbf s^T\mathbf z\ne0$, the second is strictly positive. If $\mathbf s^T\mathbf z=0$, substitution gives $\mathbf w=\mathbf z-\rho\mathbf y(\mathbf s^T\mathbf z)=\mathbf z\ne0$, making the first strictly positive. These cases cover every nonzero $\mathbf z$, proving $\mathbf H^+\succ0$. Its inverse has reciprocal positive eigenvalues and is also positive definite. In exact arithmetic the inverse update equals the inverse of the direct BFGS update.

A dense inverse update still stores $O(n^2)$ numbers and its matrix-vector step costs $O(n^2)$; avoiding a solve is not avoiding a matrix. A Cholesky-factor update is another option: update the factor and use triangular solves instead of refactorizing a full new matrix. Finite precision still matters. For the existing Hilbert-matrix conditioning demonstration, see [Linear Algebra Review and SciPy Basics](./Math-Primer.ipynb), section D2. The example is not duplicated here. Positive definiteness maintained algebraically does not by itself establish safe conditioning in floating point.

Weighted least-change characterizations distinguish the updates: an appropriate weighted Frobenius formulation for the Hessian gives DFP; the corresponding inverse formulation gives BFGS. The norm and weighting hypotheses are essential (Nocedal and Wright, Eqs. (6.9)–(6.17), pp. 138–140). The broader Broyden family and its parameterization are discussed by Al-Baali, Spedicato and Maggioni (2014), pp. 942–943; BFGS is the zero-parameter member in their convention.

Sources: Biegler (2010), Eqs. (3.18)–(3.21), pp. 44–45, and the errata for Eq. (3.23) if using its alternative rank-two vectors; Nocedal and Wright (2006), pp. 137–141; Dennis and Schnabel (1996), Lemma 9.2.1, p. 199, and Eq. (9.2.10), p. 201. These are mathematical reference formulas; the BFGS programming exercise above remains an exercise.

![The two BFGS corrections cancel the old matrix action and leave the measured gradient difference](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-bfgs-secant.png)

![Powell damping changes the secant measurement to enforce positive measured curvature](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-powell-damping.png)

![The zero sets of the two nonnegative quadratic-form terms intersect only at the origin](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-bfgs-nullspaces.png)


(qn-ref-directional)=
## Reference: directional accuracy and superlinear convergence

For a nonterminated convergent sequence $\mathbf x^k\to\mathbf x^*$ with $e_k=\|\mathbf x^k-\mathbf x^*\|>0$, Q-superlinear convergence means

$$\forall\epsilon>0,\ \exists k_0\in\mathbb N:\quad\forall k\ge k_0,\quad e_{k+1}\le\epsilon e_k.$$

Equivalently, $e_{k+1}/e_k\to0$. Finite termination requires no quotient. Linear and quadratic rates have their primary definitions in the [Newton reference](https://ndcbe.github.io/optimization/notebooks/6/newton-methods/#nu-ref-rates).

![Illustrative linear, superlinear and quadratic error sequences](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/convergence-rates.png)

Let $f\in C^2(D)$ on an open neighborhood $D$ of $\mathbf x^*$, $\nabla f(\mathbf x^*)=0$, and $\nabla^2f(\mathbf x^*)\succ0$. Suppose unit-step iterates converge to $\mathbf x^*$, with symmetric positive-definite $\mathbf B^k$, $\mathbf B^k\mathbf p^k=-\mathbf g^k$ and nonzero errors/steps before termination. Superlinear convergence is equivalent to

$$\frac{\|(\mathbf B^k-\nabla^2f(\mathbf x^k))\mathbf p^k\|}{\|\mathbf p^k\|}\to0.$$

Using $\nabla^2f(\mathbf x^*)$ instead gives the same limit by Hessian continuity. The condition concerns the directions used. For $\mathbf G=\operatorname{diag}(1,2)$, $\mathbf B=\operatorname{diag}(1,10)$ and $\mathbf p=\mathbf e_1$, the directional error is zero despite $\|\mathbf B-\mathbf G\|=8$.


![Two matrices agree on the chosen step but have different actions in an unmeasured direction](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-directional-accuracy.png)

**Proof mechanism.** Write $\mathbf e_k=\mathbf x^k-\mathbf x^*$, $\mathbf G^k=\nabla^2f(\mathbf x^k)$ and $\mathbf p_N^k=-(\mathbf G^k)^{-1}\mathbf g^k$. Hessian continuity and nondegeneracy give a bounded inverse and $\mathbf e_k+\mathbf p_N^k=o(\|\mathbf e_k\|)$. A Lipschitz Hessian strengthens this to $O(\|\mathbf e_k\|^2)$, but that stronger assumption is unnecessary for the equivalence.

Subtracting the two solves gives

$$\mathbf p^k-\mathbf p_N^k=(\mathbf G^k)^{-1}(\mathbf G^k-\mathbf B^k)\mathbf p^k.$$

Under directional accuracy, the difference is $o(\|\mathbf p^k\|)$. Hence

$$\|\mathbf e_{k+1}\|\le r_k\|\mathbf e_k\|+\eta_k\|\mathbf p^k\|,
\quad r_k,\eta_k\to0.$$

Since $\|\mathbf p^k\|\le\|\mathbf e_k\|+\|\mathbf e_{k+1}\|$, rearranging for $\eta_k<1$ gives

$$\frac{\|\mathbf e_{k+1}\|}{\|\mathbf e_k\|}\le\frac{r_k+\eta_k}{1-\eta_k}\to0.$$

Conversely, superlinear convergence gives $\mathbf p^k=-\mathbf e_k+o(\|\mathbf e_k\|)$ and thus $\|\mathbf p^k\|/\|\mathbf e_k\|\to1$. Subtracting the Newton-step remainder then gives $\mathbf p^k-\mathbf p_N^k=o(\|\mathbf e_k\|)$. Multiply by the bounded $\mathbf G^k$ and divide by $\|\mathbf p^k\|$ to recover directional accuracy. Finite termination is handled separately; the displayed ratios describe a nonterminated sequence.

A strict minimum alone is insufficient. For $f(x)=x^4$ with nonzero iterates, exact Newton has $B^k=f''(x^k)$, so its directional error is identically zero, yet $x^{k+1}=2x^k/3$ is only linear. Its limiting Hessian is zero. This exposes the missing nondegeneracy hypothesis in an unqualified reading of Biegler's Theorem 3.1; Nocedal and Wright's Theorem 3.7 states it explicitly.

For standard BFGS, the local superlinear theorem assumes convergence, a nondegenerate minimizer, the bound $\|\nabla^2f(\mathbf x)-\nabla^2f(\mathbf x^*)\|\le L\|\mathbf x-\mathbf x^*\|$ near $\mathbf x^*$ for some $L>0$, summable iterate errors, and the specified line-search/update framework. Wolfe searches test the unit step first and use a sufficient-decrease parameter strictly below $1/2$ when invoking the unit-step admissibility result. A finite gradient-norm plot does not prove those assumptions or a convergence rate. The shared Rosenbrock comparison below is one numerical experiment; its unmodified SR1 run deliberately stops on an ascent direction.

Sources: Nocedal and Wright (2006), Theorems 3.6–3.7, pp. 46–47; Assumption 6.2, p. 156, and Theorem 6.6, p. 158; Biegler (2010), Theorems 3.1–3.2, pp. 42 and 44, read with the explicit nondegenerate-minimum hypothesis.


(qn-ref-costs)=
## Reference: storage and derivative-acquisition costs

Full dense BFGS stores $n^2$ values. Fixed-memory L-BFGS retains $m$ recent secant pairs, or $2mn$ values, plus workspace and base scaling. Its two-loop recursion computes the inverse action without forming an $n\times n$ matrix (Nocedal and Wright, Algorithm 7.4, pp. 177–178). At float64 precision, $n=10^6$, $m=10$ means $8$ TB for one dense matrix versus $160$ MB for the pairs, in decimal units. This is storage accounting, not a timing benchmark.

Discarding pairs means fixed-memory L-BFGS does not inherit the full BFGS superlinear theorem. Linear-rate results for limited memory impose additional smoothness, curvature and algorithm hypotheses; memory size alone establishes no universal rate. See Liu and Nocedal (1989), Algorithm 2.1, p. 506, and Theorem 7.1, p. 524, with the discussion on p. 526.

When only function values are available, derivative cost depends on the stencil and reuse policy. Forward differences reuse $f(\mathbf x)$ and require $n+1$ distinct points for a gradient. A complete forward second-difference Hessian also needs $\mathbf x+2h\mathbf e_i$ and $\mathbf x+h\mathbf e_i+h\mathbf e_j$ for $i<j$:

$$f_{ij}\approx\frac{f(\mathbf x+h\mathbf e_i+h\mathbf e_j)-f(\mathbf x+h\mathbf e_i)-f(\mathbf x+h\mathbf e_j)+f(\mathbf x)}{h^2}.$$

Including the reused points gives $1+2n+n(n-1)/2$ evaluations, or $861$ at $n=40$. A central Hessian stencil uses $1+2n^2$ distinct points, or $3201$ at $n=40$; its mixed entries use four sign combinations per pair. These counts assume a dense Hessian, shared points, and one common step size; sparsity, automatic derivatives, constraints, noise and domain restrictions change the practical choice. The earlier rough $n^2/2$ estimate was an order-of-growth comparison, not an exact call count.

BFGS avoids Hessian acquisition, but finite-difference gradients can have an error floor that prevents the exact-derivative asymptotic theorem from applying. The notebook's intentionally nonfinite-derivative examples remain useful checks of domain restrictions.

Additional reading: Nocedal and Wright (2006), §8.1, pp. 195–203, on derivative approximation; Biegler (2010), §3.3, pp. 42–46, on quasi-Newton implementation. Factorization/error demonstrations are kept in the existing linear-algebra reference pages rather than copied here.

![A full dense matrix stores n-squared entries while limited memory stores recent displacement and gradient-difference vectors](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/qn-memory.png)


(qn-ref-comparison)=
## Shared Rosenbrock experiment: complete settings and outcomes

The handout comparison uses the existing canonical [figure implementation](https://github.com/ndcbe/optimization/blob/main/figures/plots/quasi-newton-rates.py). Its model is

$$f(\mathbf x)=100(x_2-x_1^2)^2+(1-x_1)^2,\qquad \mathbf x^0=(-1.2,1)^T.$$

The derivatives used by the experiment are

$$\nabla f=\begin{bmatrix}-400x_1(x_2-x_1^2)-2(1-x_1)\\200(x_2-x_1^2)\end{bmatrix},\qquad
\nabla^2f=\begin{bmatrix}1200x_1^2-400x_2+2&-400x_1\\-400x_1&200\end{bmatrix}.$$

All methods use the same `armijo_step` routine, Biegler's Algorithm 3.2: test $\alpha=1$ first and halve the trial value until the Armijo condition holds with $c_1=10^{-4}$, for at most sixty trial reductions. No Goldstein or Wolfe test is applied inside this shrinking loop; the accepted step is either $1$ or within a factor $1/2$ of a trial rejected for being too long (Nocedal & Wright, p. 37). The run records the gradient norm, uses threshold $10^{-11}$ and at most sixty accepted steps. These are finite experiment settings, not the hypotheses of an unconditional convergence theorem.

Newton uses the exact Hessian, adding $(10^{-8}-\lambda_{\min}+10^{-3})\mathbf I$ if $\lambda_{\min}<10^{-8}$. BFGS begins with $\mathbf B^0=\mathbf I$ and updates only if $\mathbf s^T\mathbf y>10^{-12}$. SR1 also starts from $\mathbf I$, permits either denominator sign, and stops if $|\mathbf u^T\mathbf s|<10^{-14}$; it has no curvature repair. Before searching, the implementation rejects a nonnegative slope for its BFGS/SR1 solve. Steepest descent uses $\mathbf p=-\mathbf g$.

| Method | Accepted steps | Final stored plot gradient norm | Outcome |
| --- | ---: | ---: | --- |
| Modified Newton | 22 | 1e-16 (display floor) | gradient threshold reached |
| BFGS | 37 | 1.82305e-13 | gradient threshold reached |
| SR1 | 3 | 14.1664 | ascent direction rejected |
| Steepest descent | 60 | 0.146979 | iteration cap reached |

The stored plot history floors values at $10^{-16}$ to support logarithmic display; it does not assert an exact derivative value of $10^{-16}$. The shared SR1 run's negative-denominator update creates indefinite curvature (eigenvalues approximately $-642$ and $6$); the next slope is approximately $+0.26$, so the descent-based routine rejects that direction. The norm trace remains finite and does not certify a minimum or an asymptotic rate. Source code, settings and this table are available together so the comparison can be reproduced without changing the implementation's ownership.

![Finite Rosenbrock residual histories and the intentional unmodified SR1 ascent failure](https://raw.githubusercontent.com/ndcbe/optimization/main/media/figures/quasi-newton-rates.png)
